In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="11-people-you-may-know",
    notebook_name="04_interview_walkthrough.ipynb"
)

# 04 -- People You May Know: Complete Mock Interview Walkthrough

**Goal:** Simulate a full 45-minute ML system design interview for PYMK.

---

## How This Works

This notebook simulates a real interview with:
- **Interviewer** questions and reactions
- **Strong candidate** answers with timing guidance
- **Key phrases** that demonstrate staff-level thinking
- **Common pitfalls** to avoid
- **Follow-up questions** with answers

### Timing Guide

| Step | Time | Topics |
|------|------|--------|
| 1. Clarify requirements | 2-3 min | Business goal, scale, symmetry |
| 2. Frame as ML task | 2-3 min | Edge prediction vs pointwise LTR |
| 3. Data pipeline | 3-4 min | Users, connections, interactions |
| 4. Feature engineering | 5-7 min | Graph features, user features, affinity |
| 5. Model selection | 4-5 min | GNN, explain why |
| 6. Training | 3-4 min | Temporal split, positive/negative labels |
| 7. Evaluation | 2-3 min | ROC-AUC, mAP, requests accepted |
| 8. Serving | 5-7 min | FoF, batch prediction, two pipelines |
| Follow-ups | 5-10 min | Cold start, privacy, diversity |

---

## Step 1: Clarify Requirements (2-3 minutes)

> **INTERVIEWER:** "Design a People You May Know feature, similar to LinkedIn's."

### Strong Candidate Response:

*"Great, I love this problem. Let me clarify a few things first."*

*"The motivation is to help users **discover connections and grow their network**, correct?"*

> **INTERVIEWER:** "Yes, that's right."

*"Key factors for determining suggestions would include education, work experience, and **social context** -- meaning mutual connections. Is that correct?"*

> **INTERVIEWER:** "Yes, those are the most important factors."

*"On LinkedIn, friendship is **symmetrical** -- both sides must accept. This is different from Twitter where follows are one-directional. Can I confirm this?"*

> **INTERVIEWER:** "Correct, connections require both parties to agree."

*"For scale: roughly how many total users, and how many daily active? Also, what is the average number of connections per user?"*

> **INTERVIEWER:** "About 1 billion total users, 300 million daily active, 1,000 connections on average."

*"One last question: I would assume the social graph is **relatively static** -- connections do not change dramatically day to day. This has implications for whether we use batch or real-time prediction."*

> **INTERVIEWER:** "Excellent observation. Yes, that's a reasonable assumption."

### Why This Is Strong:
- Immediately identifies the symmetry constraint (many candidates miss this)
- Proactively raises the graph stability insight (leads to batch prediction later)
- Shows understanding of the difference between LinkedIn (symmetric) and Twitter (asymmetric)

In [ ]:
requirements = {
    "Business Objective": "Help users discover connections and grow their network",
    "Key Factors": "Education, work experience, social context (mutual connections)",
    "Friendship Type": "Symmetrical (both must accept)",
    "Total Users": "~1 billion",
    "Daily Active Users": "~300 million",
    "Avg Connections": "~1,000 per user",
    "Graph Dynamics": "Relatively static (stable over short periods)",
    "Key Insight": "92% of new friendships form via friends-of-friends (Meta research)",
}

print("PYMK REQUIREMENTS SUMMARY")
print("=" * 55)
for key, value in requirements.items():
    print(f"  {key}: {value}")

---

## Step 2: Frame as ML Task (2-3 minutes)

*"The ML objective is to **maximize formed connections** between users. This directly maps to the business goal of network growth."*

*"The system takes a user as input and outputs a ranked list of potential connections."*

*"There are two approaches I would consider:"*

### Approach 1: Pointwise LTR
*"Take two user profiles, feed them into a classifier, predict P(connection). Simple but ignores graph context entirely."*

### Approach 2: Edge Prediction (Recommended)
*"Use the entire social graph as input. Predict the probability of an edge between two nodes. This captures the crucial social context -- mutual connections, neighborhood structure, triadic closure."*

### Key Phrase:
> *"I would use edge prediction because **social context** is the strongest signal. Consider: User A and User B share 10 mutual connections in one scenario, and zero in another. Pointwise LTR literally cannot distinguish these scenarios because it only sees profiles, not the graph. Edge prediction captures this difference."*

---

## Step 3: Data Pipeline (3-4 minutes)

*"We have three types of raw data:"*

1. **Users table** -- demographics, education (school, degree, major, years), work (company, title, industry), skills
2. **Connections table** -- user_id_1, user_id_2, timestamp
3. **Interactions table** -- connection requests, profile views, searches, messages, reactions

*"One important data challenge: **attribute standardization**. 'Computer Science' and 'CS' are the same thing. We need to standardize using predefined lists, heuristics, or ML-based clustering."*

### Training Labels (Temporal Split)

*"We use a time-based split to avoid data leakage:"*
1. Take graph snapshot at time **t** (model input)
2. Look at graph at time **t+1**
3. New edges between t and t+1 = **positive labels**
4. FoF pairs without new edges = **negative labels**

In [ ]:
import pandas as pd

# Demonstrate the temporal split for training data construction

print("TRAINING DATA CONSTRUCTION (Temporal Split)")
print("=" * 55)
print(f"""
Time t (model input)     Time t+1 (labels)
===================      =================

  A --- B                  A --- B
  |   / |                  |   / |
  |  /  |                  |  /  |
  C     D                  C --- D  <-- NEW EDGE! (positive label)
  |                        |   
  E                        E --- F  <-- NEW EDGE! (positive label)
       F                        
                           
Training examples:
  (C, D) = POSITIVE (new edge formed at t+1)
  (E, F) = POSITIVE (new edge formed at t+1)
  (A, D) = NEGATIVE (FoF at t, no edge at t+1)
  (B, E) = NEGATIVE (FoF at t, no edge at t+1)

Key: Labels come from the FUTURE, features from the PAST.
This ensures no data leakage.
""")

---

## Step 4: Feature Engineering (5-7 minutes)

*"Features fall into three categories:"*

### 1. User Features (per person)
- Demographics: age, gender, city, country
- Connection stats: number of connections, followers, pending requests
- Account age: newer accounts might be spam
- Activity level: received reactions (likes, shares, comments)

### 2. User-User Affinity Features (per pair) -- THE KEY DIFFERENTIATOR

**Education & Work Affinity:**
- Schools in common
- **Contemporaries at school** (overlapping years -- much stronger than just same school!)
- Same major
- Companies in common
- Same industry

**Social Affinity:**
- Profile visits (User A looked at User B's profile)
- **Mutual connections** (THE most important feature -- per Meta research)
- **Time-discounted mutual connections** (recent connections count more)

### 3. Graph-Based Features
- Common neighbors, Jaccard, Adamic-Adar
- GNN-learned node embeddings

### The Killer Feature: Time-Discounted Mutual Connections
> *"This is clever. If User A recently formed connections with C, D, E who all know User B, that is a much stronger signal than if those connections were 5 years old. Recent connections suggest A is actively expanding their network. Old connections suggest A already knows about B and chose NOT to connect."*

---

## Step 5: Model Selection (4-5 minutes)

*"Since we framed PYMK as edge prediction on a social graph, I would use a **Graph Neural Network (GNN)**."*

### How GNNs Work:
1. Each node starts with its feature vector (age, company, connections, etc.)
2. Each edge has affinity features (mutual connections, profile visits, etc.)
3. The GNN performs **message passing** -- each node aggregates information from neighbors
4. After K rounds, each node has an embedding that captures its K-hop neighborhood
5. To predict if two users will connect: **dot product** of their embeddings -> sigmoid

### GNN Variants:
- **GCN** -- basic, mean aggregation, full-batch
- **GraphSAGE** -- samples neighbors, supports mini-batch (BEST FOR PRODUCTION)
- **GAT** -- attention-weighted aggregation (more expressive)

### Key Phrase:
> *"I would use GraphSAGE for production because it supports mini-batch training via neighborhood sampling, which is essential at our 1-billion-node scale. GCN requires the full graph in memory, which is impractical."*

---

## Step 6: Evaluation (2-3 minutes)

### Offline Metrics

1. **ROC-AUC** -- for the GNN model (binary classification: connect or not)
2. **mAP** -- for the PYMK system (ranking quality with binary relevance)

*"I choose mAP over nDCG because the outcome is binary -- the user either connects or does not. There are no graded relevance levels."*

### Online Metrics

1. **Connection requests sent** -- measures outreach
   - *"Drawback: users might spam requests that never get accepted."*
2. **Connection requests accepted** -- THE NORTH STAR METRIC
   - *"A connection only forms when BOTH sides agree. This is the true measure of network growth, which is our business objective."*

---

## Step 7: Serving Architecture (5-7 minutes)

### The Efficiency Challenge

*"With 1 billion users, comparing every pair would require 10^18 comparisons -- completely impractical. We need two key optimizations:"*

### Optimization 1: Friends-of-Friends (FoF)
*"According to Meta research, **92% of new friendships form via FoF**. With 1,000 connections on average, FoF gives us 1,000 x 1,000 = 1 million candidates per user. This is a 1,000x reduction from 1 billion."*

### Optimization 2: Batch Prediction
*"Since the social graph is stable, I recommend **batch prediction** over real-time:"*
- Pre-compute PYMK for all active users
- Store results in a database
- Serve instantly when a user opens the app
- Re-compute every 7 days (established users) or daily (new users)

### Two-Pipeline Architecture

**Pipeline 1: PYMK Generation (Offline)**
```
User -> FoF Service -> 1M candidates -> Scoring (GNN) -> Top 50 -> Database
```

**Pipeline 2: Online Serving**
```
User Request -> Check DB -> Pre-computed PYMK found? -> Return
                                        No? -> Trigger generation pipeline
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('PYMK End-to-End Pipeline', fontsize=14, fontweight='bold')

steps = [
    (0.3, 1.0, '1B\nUsers', '#FFE0B2'),
    (2.3, 1.0, 'FoF\n(2-hop)\n1M cands', '#B3E5FC'),
    (4.3, 1.0, 'Light\nRanker\n1K cands', '#C8E6C9'),
    (6.3, 1.0, 'Full GNN\nScoring\n50 cands', '#F8BBD0'),
    (8.3, 1.0, 'Privacy\nFilter', '#D1C4E9'),
    (10.3, 1.0, 'Diversity\nRe-rank', '#FFF9C4'),
    (12.3, 1.0, 'Show\nTop 10', '#FFCCBC'),
]

for x, y, text, color in steps:
    box = FancyBboxPatch((x, y), 1.7, 1.8, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 0.85, y + 0.9, text, ha='center', va='center', fontsize=8, fontweight='bold')

for i in range(len(steps) - 1):
    x1 = steps[i][0] + 1.7
    x2 = steps[i+1][0]
    ax.annotate('', xy=(x2, 1.9), xytext=(x1, 1.9),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Reduction labels
reductions = ['1000x', '1000x', '20x', '', '', '']
for i, red in enumerate(reductions):
    if red:
        x = (steps[i][0] + 1.7 + steps[i+1][0]) / 2
        ax.text(x + 0.1, 3.2, red, ha='center', fontsize=8, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Common Follow-Up Questions

### Q1: "How would you handle cold-start users?"

> *"New users have no connections, so graph features are unavailable. I would use **content-based features** instead:*
> 1. *Same school or company (strongest cold-start signal)*
> 2. *Same city or industry*
> 3. *Phone contacts (with explicit user consent)*
> 4. *Popular users in the same geographic area (exploration)*
> 
> *As the user forms their first connections, we gradually transition from content-based to graph-based features. I would also re-compute suggestions daily for new users instead of weekly."*

### Q2: "How do you avoid creepy recommendations?"

> *"This is critical for user trust. Key rules:*
> 1. *Never reveal that someone viewed your profile (no 'X looked at your profile' in explanations)*
> 2. *Never use phone contacts without explicit consent*
> 3. *Respect blocks bidirectionally -- if A blocks B, neither sees the other*
> 4. *Don't suggest recently disconnected pairs (they disconnected for a reason)*
> 5. *Age-gate: additional restrictions for minors"*

### Q3: "How would you add diversity to suggestions?"

> *"I would add a **re-ranking layer** after scoring. Using Maximal Marginal Relevance (MMR), I would iteratively select candidates that are both relevant AND different from those already selected. For example, if we already picked 3 Googlers, penalize the next Googler candidate to make room for someone from a different company."*

### Q4: "How do you handle scale -- 1 billion users?"

> *"Three strategies:*
> 1. *FoF narrowing: 1B -> 1M candidates (1000x reduction)*
> 2. *Batch prediction: pre-compute for active users, store in DB*
> 3. *Graph partitioning: distribute the graph across machines, use GraphSAGE which supports mini-batch training via neighbor sampling"*

### Q5: "What is the single most important feature?"

> *"**Mutual connections** -- hands down. Meta's research showed that 92% of new friendships form between friends-of-friends. The number of mutual connections is consistently the strongest predictor in link prediction tasks. Second would be time-discounted mutual connections, which distinguishes actively growing networks from stable ones."*

In [ ]:
# === COMPLETE INTERVIEW CHEAT SHEET ===

cheat_sheet = """
====================================================================
        PEOPLE YOU MAY KNOW -- INTERVIEW CHEAT SHEET
====================================================================

30-SECOND PITCH:
"PYMK predicts potential connections by formulating it as edge 
prediction on the social graph. The strongest signal is mutual 
connections -- 92% of new friendships come from FoF. I'd use a 
GNN (GraphSAGE for scalability) with graph features (common 
neighbors, Adamic-Adar, time-discounted mutual connections) and 
profile affinity features (same school, company, industry). For 
evaluation: ROC-AUC and mAP offline, connection requests accepted
online. Serving uses batch prediction with a two-pipeline 
architecture: offline FoF + GNN scoring, online DB lookup."

KEY NUMBERS:
  - 92% of new connections via FoF (Meta research)
  - FoF reduces search: 1B -> 1M (1000x reduction)
  - Avg 1,000 connections per user
  - Re-compute every 7 days (established) / daily (new users)
  - Pre-compute ~100 suggestions, show ~10 per visit

VOCABULARY TO USE:
  - "Edge prediction" / "Link prediction"
  - "Friends-of-friends" / "2-hop BFS"
  - "Triadic closure" (triangles tend to close in social networks)
  - "Message passing" / "Neighborhood aggregation" (how GNNs work)
  - "Time-discounted mutual connections"
  - "Symmetrical connections" (both must accept)
  - "Batch prediction" (graph is stable)
  - "North star metric: connection requests ACCEPTED"

WHAT MAKES THIS DIFFERENT FROM OTHER REC SYSTEMS:
  - Fundamentally a GRAPH problem (not matrix factorization)
  - Bidirectional (both sides must agree)
  - Graph features dominate profile features
  - Batch prediction is preferred (stable graph)
  - Privacy is especially critical (creepy factor)

PITFALLS TO AVOID:
  x Using collaborative filtering (this is a GRAPH problem)
  x Forgetting about FoF optimization (must mention for scale)
  x Not mentioning privacy (blocks, profile view secrecy)
  x Suggesting real-time prediction (too expensive at 1B scale)
  x Using only profile features (graph features are much stronger)
  x Forgetting reciprocity (both sides must want to connect)
====================================================================
"""

print(cheat_sheet)

## Key Takeaways

1. **PYMK is fundamentally a graph problem** -- use edge prediction, not collaborative filtering

2. **Friends-of-Friends is the #1 strategy** -- 92% of new connections come from FoF; also reduces search space 1000x

3. **Mutual connections is the strongest feature** -- combined with time-discounting for extra credit

4. **Edge prediction > pointwise LTR** -- graph context (mutual friends, neighborhood) is essential

5. **GraphSAGE for production** -- supports mini-batch training via neighbor sampling

6. **Batch prediction preferred** -- social graphs are stable; pre-compute and serve from DB

7. **North star = connection requests accepted** -- not sent (avoids gaming by spammers)

8. **Privacy is non-negotiable** -- never reveal viewers, respect blocks, consent for contacts

9. **Two-pipeline architecture** -- offline generation (FoF + GNN) + online serving (DB lookup)

10. **Cold-start via content matching** -- same school/company until graph features are available

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)